In [1]:
import pandas as pd

data = pd.read_csv("Twitter_Data.csv")
data.head()

,clean_text,category
0,when modi promised “minimum government maximum...,-1.0
1,talk all the nonsense and continue all the dra...,0.0
2,what did just say vote for modi welcome bjp t...,1.0
3,asking his supporters prefix chowkidar their n...,1.0
4,answer who among these the most powerful world...,1.0


In [2]:
data['category'] = data['category'].map({
    -1: 0,
    0: 1,
    1: 2
})

In [3]:
data.info()
data['category'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162980 entries, 0 to 162979
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   clean_text  162976 non-null  object 
 1   category    162973 non-null  float64
dtypes: float64(1), object(1)
memory usage: 2.5+ MB


category
2.0    72250
1.0    55213
0.0    35510
Name: count, dtype: int64

In [4]:
data.drop_duplicates(inplace=True)
data.shape

(162979, 2)

removing html tags bcz imdb stores the line breaking and other entities as tags to provide better result

In [5]:
data['clean_text'].head()

0    when modi promised “minimum government maximum...
1    talk all the nonsense and continue all the dra...
2    what did just say vote for modi  welcome bjp t...
3    asking his supporters prefix chowkidar their n...
4    answer who among these the most powerful world...
Name: clean_text, dtype: object

In [6]:
# Drop rows where clean_text is missing
data.dropna(subset=['clean_text'], inplace=True)

Data cleaning

In [7]:
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

ps = PorterStemmer()

def cleaning_text(text):
    #lowercase
    text = text.lower()
    #Punctuation
    text = ''.join([c for c in text if c not in string.punctuation])
    words = text.split()
    stop_words = set(stopwords.words('english'))
    stop_words.discard('not')
    stop_words.discard('no')
    stop_words.discard('nor')

    words = [ps.stem(w) for w in words if w not in stop_words]
    return ' '.join(words)

data['cleaned_text'] = data['clean_text'].apply(cleaning_text)


In [8]:
print(data['category'].value_counts())

category
2.0    72249
1.0    55211
0.0    35509
Name: count, dtype: int64


Outlier handdling

In [9]:
data['length'] = data['clean_text'].apply(len)
data = data[data['length'] > 20]

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(data['cleaned_text']).toarray()

y = data['category'].values

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
data = data.dropna(subset=['clean_text', 'category'])
data['length'] = data['clean_text'].astype(str).apply(len)
data = data[data['length'] > 20]

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2)
)
X = tfidf.fit_transform(data['cleaned_text']) 
y = data['category'].values

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

import numpy as np
print(f"Any NaNs in y_train? {np.isnan(y_train).any()}")

Any NaNs in y_train? False


In [13]:
# Remove any rows where 'category' or 'cleaned_text' is missing
data.dropna(subset=['category', 'cleaned_text'], inplace=True)

# Double check that there are no NaNs left
print(data.isnull().sum())

clean_text      0
category        0
cleaned_text    0
length          0
dtype: int64


model

In [14]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [15]:
#Accuracy
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8409355254471723
              precision    recall  f1-score   support

         0.0       0.76      0.82      0.79      7137
         1.0       0.81      0.91      0.85     10470
         2.0       0.92      0.80      0.86     14204

    accuracy                           0.84     31811
   macro avg       0.83      0.84      0.83     31811
weighted avg       0.85      0.84      0.84     31811



In [16]:
def predict_sentiment_debug(text):
    text = cleaning_text(text)
    vector = tfidf.transform([text])
    probs = model.predict_proba(vector)[0]
    return {
        "Negative": probs[0],
        "Neutral": probs[1],
        "Positive": probs[2]
    }
predict_sentiment_debug("This is not good")

{'Negative': np.float64(0.9336849846054471),
 'Neutral': np.float64(0.0012701418231695683),
 'Positive': np.float64(0.06504487357138317)}

In [ ]:
#udf
def predict_sentiment(text):
    text = cleaning_text(text)
    vector = tfidf.transform([text])
    result = model.predict(vector)
    if result[0] == 2 : return "Positive" 
    elif result[0] == 1 : return "Neutral"
    else: return "Negative"
print(predict_sentiment("This is a fantastic initiative that will help everyone."))
print(predict_sentiment("The policy is very bad."))
print(predict_sentiment("The meeting is tomorrow."))

Positive
Negative
Neutral


In [20]:
import pickle

model_data = {
    "vectorizer": tfidf,
    "model": model
}

with open("sentiment_model.pkl", "wb") as f:
    pickle.dump(model_data, f)

print("Model + vectorizer saved!")

Model + vectorizer saved!


In [46]:
from sklearn.base import BaseEstimator, TransformerMixin

class TextCleaner(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [cleaning_text(text) for text in X]


In [63]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("cleaner", TextCleaner()),
    ("vectorizer", tfidf),
    ("classifier", model)
])

In [ ]:
pipeline.fit_transform(X_train, y_train)

AttributeError: 'csr_matrix' object has no attribute 'lower'

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('tfidf', tfidf),
    ('model', model)
])

import pickle
with open("sentiment_model.pkl", "wb") as f:
    pickle.dump(pipeline, f)

In [ ]:
print(predict_sentiment("The story is amazing."))

Positive


In [ ]:
print(data['category'].unique())

[0. 1. 2.]


In [ ]:
pipeline.predict([cleaning_text("The story is amazing.")])

array([2.])